# Task 4: Multi-Agent System

This notebook demonstrates an agentic workflow with two specialized agents:
- **Math Agent**: Handles mathematical operations (arithmetic, statistics, etc.)
- **String Agent**: Handles string manipulations (transformations, analysis, etc.)

## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                    Orchestrator                         │
│         (Routes tasks to appropriate agents)            │
└───────────────────┬─────────────────────────────────────┘
                    │
        ┌───────────┴───────────┐
        │                       │
        ▼                       ▼
┌───────────────┐       ┌───────────────┐
│  Math Agent   │       │ String Agent  │
│               │       │               │
│ - add         │       │ - uppercase   │
│ - subtract    │       │ - lowercase   │
│ - multiply    │       │ - reverse     │
│ - divide      │       │ - count_chars │
│ - power       │       │ - replace     │
│ - sqrt        │       │ - split       │
│ - mean        │       │ - join        │
│ - factorial   │       │ - palindrome  │
└───────────────┘       └───────────────┘
```

## 1. Setup and Imports

In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional, Union
from dataclasses import dataclass, field
from enum import Enum
import math
import re
from datetime import datetime

## 2. Core Agent Framework

Define the base classes and data structures for our multi-agent system.

In [ ]:
class TaskStatus(Enum):
    """Status of a task in the multi-agent system."""
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"


@dataclass
class Task:
    """Represents a task to be processed by an agent."""
    task_id: str
    operation: str
    arguments: Dict[str, Any]
    status: TaskStatus = TaskStatus.PENDING
    result: Any = None
    error: Optional[str] = None
    agent_name: Optional[str] = None
    created_at: datetime = field(default_factory=datetime.now)
    completed_at: Optional[datetime] = None


@dataclass
class AgentResponse:
    """Response from an agent after processing a task."""
    success: bool
    result: Any
    message: str
    agent_name: str
    operation: str

In [ ]:
class BaseAgent(ABC):
    """Abstract base class for all agents in the system."""
    
    def __init__(self, name: str):
        self.name = name
        self.operations: Dict[str, callable] = {}
        self._register_operations()
    
    @abstractmethod
    def _register_operations(self) -> None:
        """Register all operations this agent can perform."""
        pass
    
    @abstractmethod
    def get_capabilities(self) -> List[str]:
        """Return list of operations this agent can perform."""
        pass
    
    def can_handle(self, operation: str) -> bool:
        """Check if this agent can handle a specific operation."""
        return operation.lower() in self.operations
    
    def execute(self, task: Task) -> AgentResponse:
        """Execute a task and return the response."""
        operation = task.operation.lower()
        
        if not self.can_handle(operation):
            return AgentResponse(
                success=False,
                result=None,
                message=f"Operation '{operation}' not supported by {self.name}",
                agent_name=self.name,
                operation=operation
            )
        
        try:
            result = self.operations[operation](**task.arguments)
            return AgentResponse(
                success=True,
                result=result,
                message=f"Successfully executed '{operation}'",
                agent_name=self.name,
                operation=operation
            )
        except Exception as e:
            return AgentResponse(
                success=False,
                result=None,
                message=f"Error executing '{operation}': {str(e)}",
                agent_name=self.name,
                operation=operation
            )
    
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(name='{self.name}', operations={list(self.operations.keys())})"

## 3. Math Agent

The Math Agent handles all mathematical operations including arithmetic, statistics, and advanced math functions.

In [ ]:
class MathAgent(BaseAgent):
    """Agent specialized in mathematical operations."""
    
    def __init__(self):
        super().__init__("MathAgent")
    
    def _register_operations(self) -> None:
        """Register all mathematical operations."""
        self.operations = {
            # Basic arithmetic
            "add": self._add,
            "subtract": self._subtract,
            "multiply": self._multiply,
            "divide": self._divide,
            
            # Advanced operations
            "power": self._power,
            "sqrt": self._sqrt,
            "factorial": self._factorial,
            "abs": self._abs,
            "mod": self._mod,
            
            # Statistical operations
            "mean": self._mean,
            "median": self._median,
            "sum": self._sum,
            "min": self._min,
            "max": self._max,
            "variance": self._variance,
            "std_dev": self._std_dev,
            
            # Trigonometric operations
            "sin": self._sin,
            "cos": self._cos,
            "tan": self._tan,
            
            # Logarithmic operations
            "log": self._log,
            "log10": self._log10,
            "exp": self._exp,
        }
    
    def get_capabilities(self) -> List[str]:
        """Return list of mathematical operations."""
        return [
            "Arithmetic: add, subtract, multiply, divide, mod",
            "Advanced: power, sqrt, factorial, abs",
            "Statistics: mean, median, sum, min, max, variance, std_dev",
            "Trigonometry: sin, cos, tan",
            "Logarithmic: log, log10, exp"
        ]
    
    # Basic arithmetic operations
    def _add(self, a: float, b: float) -> float:
        return a + b
    
    def _subtract(self, a: float, b: float) -> float:
        return a - b
    
    def _multiply(self, a: float, b: float) -> float:
        return a * b
    
    def _divide(self, a: float, b: float) -> float:
        if b == 0:
            raise ValueError("Division by zero is not allowed")
        return a / b
    
    # Advanced operations
    def _power(self, base: float, exponent: float) -> float:
        return math.pow(base, exponent)
    
    def _sqrt(self, value: float) -> float:
        if value < 0:
            raise ValueError("Cannot compute square root of negative number")
        return math.sqrt(value)
    
    def _factorial(self, n: int) -> int:
        if n < 0:
            raise ValueError("Factorial is not defined for negative numbers")
        return math.factorial(int(n))
    
    def _abs(self, value: float) -> float:
        return abs(value)
    
    def _mod(self, a: float, b: float) -> float:
        return a % b
    
    # Statistical operations
    def _mean(self, numbers: List[float]) -> float:
        if not numbers:
            raise ValueError("Cannot compute mean of empty list")
        return sum(numbers) / len(numbers)
    
    def _median(self, numbers: List[float]) -> float:
        if not numbers:
            raise ValueError("Cannot compute median of empty list")
        sorted_nums = sorted(numbers)
        n = len(sorted_nums)
        mid = n // 2
        if n % 2 == 0:
            return (sorted_nums[mid - 1] + sorted_nums[mid]) / 2
        return sorted_nums[mid]
    
    def _sum(self, numbers: List[float]) -> float:
        return sum(numbers)
    
    def _min(self, numbers: List[float]) -> float:
        return min(numbers)
    
    def _max(self, numbers: List[float]) -> float:
        return max(numbers)
    
    def _variance(self, numbers: List[float]) -> float:
        if len(numbers) < 2:
            raise ValueError("Variance requires at least 2 numbers")
        mean = self._mean(numbers)
        return sum((x - mean) ** 2 for x in numbers) / (len(numbers) - 1)
    
    def _std_dev(self, numbers: List[float]) -> float:
        return math.sqrt(self._variance(numbers))
    
    # Trigonometric operations
    def _sin(self, angle: float) -> float:
        return math.sin(math.radians(angle))
    
    def _cos(self, angle: float) -> float:
        return math.cos(math.radians(angle))
    
    def _tan(self, angle: float) -> float:
        return math.tan(math.radians(angle))
    
    # Logarithmic operations
    def _log(self, value: float, base: float = math.e) -> float:
        if value <= 0:
            raise ValueError("Logarithm is not defined for non-positive numbers")
        return math.log(value, base)
    
    def _log10(self, value: float) -> float:
        if value <= 0:
            raise ValueError("Logarithm is not defined for non-positive numbers")
        return math.log10(value)
    
    def _exp(self, value: float) -> float:
        return math.exp(value)

## 4. String Agent

The String Agent handles all string manipulation operations including transformations, analysis, and formatting.

In [ ]:
class StringAgent(BaseAgent):
    """Agent specialized in string manipulation operations."""
    
    def __init__(self):
        super().__init__("StringAgent")
    
    def _register_operations(self) -> None:
        """Register all string operations."""
        self.operations = {
            # Case transformations
            "uppercase": self._uppercase,
            "lowercase": self._lowercase,
            "capitalize": self._capitalize,
            "title_case": self._title_case,
            "swap_case": self._swap_case,
            
            # String manipulations
            "reverse": self._reverse,
            "replace": self._replace,
            "strip": self._strip,
            "split": self._split,
            "join": self._join,
            "concat": self._concat,
            "repeat": self._repeat,
            
            # String analysis
            "count_chars": self._count_chars,
            "count_words": self._count_words,
            "count_substring": self._count_substring,
            "find": self._find,
            "is_palindrome": self._is_palindrome,
            "is_numeric": self._is_numeric,
            "is_alpha": self._is_alpha,
            
            # Advanced operations
            "slice": self._slice,
            "pad_left": self._pad_left,
            "pad_right": self._pad_right,
            "remove_whitespace": self._remove_whitespace,
            "extract_numbers": self._extract_numbers,
            "extract_words": self._extract_words,
        }
    
    def get_capabilities(self) -> List[str]:
        """Return list of string operations."""
        return [
            "Case: uppercase, lowercase, capitalize, title_case, swap_case",
            "Manipulation: reverse, replace, strip, split, join, concat, repeat",
            "Analysis: count_chars, count_words, count_substring, find, is_palindrome",
            "Validation: is_numeric, is_alpha",
            "Advanced: slice, pad_left, pad_right, remove_whitespace, extract_numbers, extract_words"
        ]
    
    # Case transformations
    def _uppercase(self, text: str) -> str:
        return text.upper()
    
    def _lowercase(self, text: str) -> str:
        return text.lower()
    
    def _capitalize(self, text: str) -> str:
        return text.capitalize()
    
    def _title_case(self, text: str) -> str:
        return text.title()
    
    def _swap_case(self, text: str) -> str:
        return text.swapcase()
    
    # String manipulations
    def _reverse(self, text: str) -> str:
        return text[::-1]
    
    def _replace(self, text: str, old: str, new: str) -> str:
        return text.replace(old, new)
    
    def _strip(self, text: str, chars: Optional[str] = None) -> str:
        return text.strip(chars)
    
    def _split(self, text: str, delimiter: str = " ") -> List[str]:
        return text.split(delimiter)
    
    def _join(self, items: List[str], delimiter: str = " ") -> str:
        return delimiter.join(items)
    
    def _concat(self, *strings: str) -> str:
        return "".join(strings)
    
    def _repeat(self, text: str, times: int) -> str:
        return text * times
    
    # String analysis
    def _count_chars(self, text: str, include_spaces: bool = True) -> int:
        if include_spaces:
            return len(text)
        return len(text.replace(" ", ""))
    
    def _count_words(self, text: str) -> int:
        return len(text.split())
    
    def _count_substring(self, text: str, substring: str) -> int:
        return text.count(substring)
    
    def _find(self, text: str, substring: str) -> int:
        return text.find(substring)
    
    def _is_palindrome(self, text: str) -> bool:
        cleaned = re.sub(r'[^a-zA-Z0-9]', '', text.lower())
        return cleaned == cleaned[::-1]
    
    def _is_numeric(self, text: str) -> bool:
        return text.replace('.', '').replace('-', '').isdigit()
    
    def _is_alpha(self, text: str) -> bool:
        return text.replace(' ', '').isalpha()
    
    # Advanced operations
    def _slice(self, text: str, start: int = 0, end: Optional[int] = None) -> str:
        return text[start:end]
    
    def _pad_left(self, text: str, width: int, char: str = " ") -> str:
        return text.rjust(width, char)
    
    def _pad_right(self, text: str, width: int, char: str = " ") -> str:
        return text.ljust(width, char)
    
    def _remove_whitespace(self, text: str) -> str:
        return re.sub(r'\s+', '', text)
    
    def _extract_numbers(self, text: str) -> List[str]:
        return re.findall(r'-?\d+\.?\d*', text)
    
    def _extract_words(self, text: str) -> List[str]:
        return re.findall(r'\b[a-zA-Z]+\b', text)

## 5. Orchestrator

The Orchestrator manages the multi-agent system, routing tasks to appropriate agents and handling complex workflows.

In [ ]:
class Orchestrator:
    """Orchestrator that coordinates multiple agents and routes tasks."""
    
    def __init__(self):
        self.agents: Dict[str, BaseAgent] = {}
        self.task_history: List[Task] = []
        self._task_counter = 0
    
    def register_agent(self, agent: BaseAgent) -> None:
        """Register an agent with the orchestrator."""
        self.agents[agent.name] = agent
        print(f"Registered agent: {agent.name}")
    
    def list_agents(self) -> List[str]:
        """List all registered agents."""
        return list(self.agents.keys())
    
    def get_all_capabilities(self) -> Dict[str, List[str]]:
        """Get capabilities of all registered agents."""
        return {name: agent.get_capabilities() for name, agent in self.agents.items()}
    
    def _generate_task_id(self) -> str:
        """Generate a unique task ID."""
        self._task_counter += 1
        return f"task_{self._task_counter:04d}"
    
    def _find_agent_for_operation(self, operation: str) -> Optional[BaseAgent]:
        """Find an agent that can handle the given operation."""
        for agent in self.agents.values():
            if agent.can_handle(operation):
                return agent
        return None
    
    def execute_task(self, operation: str, **kwargs) -> AgentResponse:
        """Execute a single task by routing it to the appropriate agent."""
        # Create task
        task = Task(
            task_id=self._generate_task_id(),
            operation=operation,
            arguments=kwargs
        )
        
        # Find appropriate agent
        agent = self._find_agent_for_operation(operation)
        
        if agent is None:
            task.status = TaskStatus.FAILED
            task.error = f"No agent found for operation: {operation}"
            self.task_history.append(task)
            return AgentResponse(
                success=False,
                result=None,
                message=task.error,
                agent_name="Orchestrator",
                operation=operation
            )
        
        # Execute task
        task.status = TaskStatus.IN_PROGRESS
        task.agent_name = agent.name
        
        response = agent.execute(task)
        
        # Update task status
        task.status = TaskStatus.COMPLETED if response.success else TaskStatus.FAILED
        task.result = response.result
        task.error = None if response.success else response.message
        task.completed_at = datetime.now()
        
        self.task_history.append(task)
        return response
    
    def execute_workflow(self, tasks: List[Dict[str, Any]]) -> List[AgentResponse]:
        """Execute a sequence of tasks (workflow)."""
        results = []
        context = {}  # Store intermediate results
        
        for i, task_def in enumerate(tasks):
            operation = task_def.get("operation")
            arguments = task_def.get("arguments", {})
            
            # Replace placeholders with context values
            resolved_args = {}
            for key, value in arguments.items():
                if isinstance(value, str) and value.startswith("$"):
                    # Reference to previous result
                    ref_key = value[1:]
                    if ref_key in context:
                        resolved_args[key] = context[ref_key]
                    else:
                        resolved_args[key] = value
                else:
                    resolved_args[key] = value
            
            print(f"Step {i+1}: Executing '{operation}' with args: {resolved_args}")
            response = self.execute_task(operation, **resolved_args)
            results.append(response)
            
            # Store result in context for potential use by subsequent tasks
            context[f"step_{i+1}"] = response.result
            context["last_result"] = response.result
            
            print(f"  -> Agent: {response.agent_name}, Success: {response.success}, Result: {response.result}")
            
            if not response.success:
                print(f"  -> Workflow stopped due to error: {response.message}")
                break
        
        return results
    
    def get_task_history(self) -> List[Task]:
        """Get the history of all executed tasks."""
        return self.task_history
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get statistics about task execution."""
        total = len(self.task_history)
        completed = sum(1 for t in self.task_history if t.status == TaskStatus.COMPLETED)
        failed = sum(1 for t in self.task_history if t.status == TaskStatus.FAILED)
        
        by_agent = {}
        for task in self.task_history:
            agent = task.agent_name or "Unknown"
            if agent not in by_agent:
                by_agent[agent] = {"total": 0, "completed": 0, "failed": 0}
            by_agent[agent]["total"] += 1
            if task.status == TaskStatus.COMPLETED:
                by_agent[agent]["completed"] += 1
            elif task.status == TaskStatus.FAILED:
                by_agent[agent]["failed"] += 1
        
        return {
            "total_tasks": total,
            "completed": completed,
            "failed": failed,
            "success_rate": completed / total if total > 0 else 0,
            "by_agent": by_agent
        }

## 6. Initialize the Multi-Agent System

In [ ]:
# Create the orchestrator
orchestrator = Orchestrator()

# Create and register agents
math_agent = MathAgent()
string_agent = StringAgent()

orchestrator.register_agent(math_agent)
orchestrator.register_agent(string_agent)

In [ ]:
# Display system capabilities
print("=" * 60)
print("Multi-Agent System Capabilities")
print("=" * 60)

for agent_name, capabilities in orchestrator.get_all_capabilities().items():
    print(f"\n{agent_name}:")
    for cap in capabilities:
        print(f"  - {cap}")

## 7. Demo: Individual Operations

Demonstrate how the orchestrator routes tasks to the appropriate agents.

In [ ]:
print("=" * 60)
print("Demo: Math Agent Operations")
print("=" * 60)

# Basic arithmetic
result = orchestrator.execute_task("add", a=15, b=27)
print(f"\nadd(15, 27) = {result.result}")

result = orchestrator.execute_task("multiply", a=6, b=7)
print(f"multiply(6, 7) = {result.result}")

result = orchestrator.execute_task("divide", a=100, b=8)
print(f"divide(100, 8) = {result.result}")

# Advanced operations
result = orchestrator.execute_task("power", base=2, exponent=10)
print(f"power(2, 10) = {result.result}")

result = orchestrator.execute_task("sqrt", value=144)
print(f"sqrt(144) = {result.result}")

result = orchestrator.execute_task("factorial", n=6)
print(f"factorial(6) = {result.result}")

# Statistical operations
numbers = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
result = orchestrator.execute_task("mean", numbers=numbers)
print(f"\nmean({numbers}) = {result.result}")

result = orchestrator.execute_task("median", numbers=numbers)
print(f"median({numbers}) = {result.result}")

result = orchestrator.execute_task("std_dev", numbers=numbers)
print(f"std_dev({numbers}) = {result.result:.2f}")

# Trigonometry
result = orchestrator.execute_task("sin", angle=30)
print(f"\nsin(30°) = {result.result:.4f}")

result = orchestrator.execute_task("cos", angle=60)
print(f"cos(60°) = {result.result:.4f}")

In [ ]:
print("=" * 60)
print("Demo: String Agent Operations")
print("=" * 60)

# Case transformations
text = "Hello, Multi-Agent System!"
print(f"\nOriginal: '{text}'")

result = orchestrator.execute_task("uppercase", text=text)
print(f"uppercase: '{result.result}'")

result = orchestrator.execute_task("lowercase", text=text)
print(f"lowercase: '{result.result}'")

result = orchestrator.execute_task("title_case", text="the quick brown fox")
print(f"title_case('the quick brown fox'): '{result.result}'")

# String manipulations
result = orchestrator.execute_task("reverse", text="Python")
print(f"\nreverse('Python'): '{result.result}'")

result = orchestrator.execute_task("replace", text="Hello World", old="World", new="Agents")
print(f"replace('Hello World', 'World', 'Agents'): '{result.result}'")

result = orchestrator.execute_task("split", text="apple,banana,cherry", delimiter=",")
print(f"split('apple,banana,cherry', ','): {result.result}")

result = orchestrator.execute_task("join", items=["one", "two", "three"], delimiter=" - ")
print(f"join(['one', 'two', 'three'], ' - '): '{result.result}'")

result = orchestrator.execute_task("repeat", text="abc", times=4)
print(f"repeat('abc', 4): '{result.result}'")

# String analysis
print(f"\nString Analysis:")
result = orchestrator.execute_task("count_words", text="The quick brown fox jumps over the lazy dog")
print(f"count_words('The quick brown fox jumps over the lazy dog'): {result.result}")

result = orchestrator.execute_task("is_palindrome", text="A man a plan a canal Panama")
print(f"is_palindrome('A man a plan a canal Panama'): {result.result}")

result = orchestrator.execute_task("is_palindrome", text="Hello World")
print(f"is_palindrome('Hello World'): {result.result}")

# Advanced operations
print(f"\nAdvanced Operations:")
result = orchestrator.execute_task("extract_numbers", text="Order #123 has 5 items for $99.99")
print(f"extract_numbers('Order #123 has 5 items for $99.99'): {result.result}")

result = orchestrator.execute_task("pad_left", text="42", width=5, char="0")
print(f"pad_left('42', 5, '0'): '{result.result}'")

## 8. Demo: Complex Workflows

Demonstrate how to chain multiple operations together in workflows that span multiple agents.

In [ ]:
print("=" * 60)
print("Workflow 1: Text Analysis with Statistics")
print("=" * 60)
print("\nAnalyze a sentence and compute statistics about it.\n")

# Define the workflow
workflow1 = [
    {"operation": "count_words", "arguments": {"text": "The quick brown fox jumps over the lazy dog"}},
    {"operation": "count_chars", "arguments": {"text": "The quick brown fox jumps over the lazy dog", "include_spaces": False}},
    {"operation": "extract_words", "arguments": {"text": "The quick brown fox jumps over the lazy dog"}},
]

results = orchestrator.execute_workflow(workflow1)

# Now use the math agent to compute statistics on word lengths
words = results[2].result
word_lengths = [len(word) for word in words]
print(f"\nWord lengths: {word_lengths}")

mean_result = orchestrator.execute_task("mean", numbers=word_lengths)
print(f"Mean word length: {mean_result.result:.2f}")

max_result = orchestrator.execute_task("max", numbers=word_lengths)
print(f"Max word length: {max_result.result}")

In [ ]:
print("=" * 60)
print("Workflow 2: Data Processing Pipeline")
print("=" * 60)
print("\nExtract numbers from text, compute statistics, format result.\n")

# Step 1: Extract numbers from text
text_with_numbers = "Sales: Q1=$1500, Q2=$2300, Q3=$1800, Q4=$2900"
extract_result = orchestrator.execute_task("extract_numbers", text=text_with_numbers)
print(f"Extracted numbers: {extract_result.result}")

# Step 2: Convert to floats and compute statistics
numbers = [float(n) for n in extract_result.result]
print(f"Numbers as floats: {numbers}")

sum_result = orchestrator.execute_task("sum", numbers=numbers)
mean_result = orchestrator.execute_task("mean", numbers=numbers)
max_result = orchestrator.execute_task("max", numbers=numbers)
min_result = orchestrator.execute_task("min", numbers=numbers)

print(f"\nStatistics:")
print(f"  Total: ${sum_result.result:.2f}")
print(f"  Average: ${mean_result.result:.2f}")
print(f"  Highest: ${max_result.result:.2f}")
print(f"  Lowest: ${min_result.result:.2f}")

# Step 3: Format the summary
summary = f"Total: {sum_result.result}, Avg: {mean_result.result:.2f}"
uppercase_result = orchestrator.execute_task("uppercase", text=summary)
print(f"\nFormatted Summary: {uppercase_result.result}")

In [ ]:
print("=" * 60)
print("Workflow 3: Chained Operations with Context")
print("=" * 60)
print("\nDemonstrate workflow with result passing.\n")

# Define a workflow where results flow between steps
workflow3 = [
    # Math: Calculate the result of (5 + 3) * 2
    {"operation": "add", "arguments": {"a": 5, "b": 3}},
    {"operation": "multiply", "arguments": {"a": "$last_result", "b": 2}},
    # String: Convert result to a padded string
    # Note: We'll need to handle this manually since operations cross agents
]

results = orchestrator.execute_workflow(workflow3)

# Continue with the final result
final_number = results[-1].result
print(f"\nFinal calculation result: {final_number}")

# Format as string
pad_result = orchestrator.execute_task("pad_left", text=str(int(final_number)), width=6, char="0")
print(f"Padded as string: '{pad_result.result}'")

## 9. System Statistics and History

In [ ]:
print("=" * 60)
print("System Statistics")
print("=" * 60)

stats = orchestrator.get_statistics()

print(f"\nTotal Tasks Executed: {stats['total_tasks']}")
print(f"Completed: {stats['completed']}")
print(f"Failed: {stats['failed']}")
print(f"Success Rate: {stats['success_rate']*100:.1f}%")

print("\nTasks by Agent:")
for agent_name, agent_stats in stats['by_agent'].items():
    print(f"  {agent_name}:")
    print(f"    Total: {agent_stats['total']}, Completed: {agent_stats['completed']}, Failed: {agent_stats['failed']}")

In [ ]:
print("=" * 60)
print("Recent Task History (Last 10)")
print("=" * 60)

history = orchestrator.get_task_history()[-10:]

for task in history:
    status_symbol = "✓" if task.status == TaskStatus.COMPLETED else "✗"
    print(f"\n{status_symbol} [{task.task_id}] {task.operation}")
    print(f"   Agent: {task.agent_name}")
    print(f"   Args: {task.arguments}")
    print(f"   Result: {task.result}")

## 10. Error Handling Demo

In [ ]:
print("=" * 60)
print("Error Handling Demonstration")
print("=" * 60)

# Try division by zero
print("\n1. Division by zero:")
result = orchestrator.execute_task("divide", a=10, b=0)
print(f"   Success: {result.success}")
print(f"   Message: {result.message}")

# Try square root of negative number
print("\n2. Square root of negative number:")
result = orchestrator.execute_task("sqrt", value=-16)
print(f"   Success: {result.success}")
print(f"   Message: {result.message}")

# Try unknown operation
print("\n3. Unknown operation:")
result = orchestrator.execute_task("unknown_operation", x=5)
print(f"   Success: {result.success}")
print(f"   Message: {result.message}")

# Try factorial of negative number
print("\n4. Factorial of negative number:")
result = orchestrator.execute_task("factorial", n=-5)
print(f"   Success: {result.success}")
print(f"   Message: {result.message}")

## Summary

This notebook demonstrated a **Multi-Agent System** with the following components:

### Architecture
1. **BaseAgent**: Abstract base class defining the agent interface
2. **MathAgent**: Specialized agent for mathematical operations
3. **StringAgent**: Specialized agent for string manipulations
4. **Orchestrator**: Coordinates agents and routes tasks

### Key Features
- **Task Routing**: Automatically routes tasks to the appropriate agent
- **Workflow Support**: Chain multiple operations with result passing
- **Error Handling**: Graceful error handling with informative messages
- **Task History**: Track all executed tasks for auditing
- **Statistics**: Monitor system performance and agent utilization

### Agents Capabilities

| Math Agent | String Agent |
|------------|-------------|
| Arithmetic (add, subtract, multiply, divide) | Case transformations |
| Advanced math (power, sqrt, factorial) | String manipulations |
| Statistics (mean, median, std_dev) | String analysis |
| Trigonometry (sin, cos, tan) | Pattern extraction |
| Logarithms (log, exp) | Validation checks |